# pandas examples

`Calendar` accepts pandas date-like inputs directly. The important point is that the top-level API remains NumPy-backed: pandas values are accepted as inputs, while outputs remain NumPy scalars or NumPy arrays: dates are `datetime64[D]` with `NaT` for missing values, counts are `float64` with `NaN` for missing values, and `isbizday` vectors are masked arrays with `fill_value=False`.

In [ ]:
import pandas as pd

from bizdays import Calendar

cal = Calendar.load(name="ANBIMA")

Pandas `DatetimeIndex` values work directly with `isbizday`.

In [ ]:
dates = pd.to_datetime(["2014-01-12", "2014-01-13", None])
cal.isbizday(dates)

Scalar pandas timestamps are accepted and still produce NumPy scalars.

In [ ]:
result = cal.following(pd.Timestamp("2015-12-25"))
result, type(result)

Vectorized business-day calculations fit naturally into a DataFrame workflow.

In [ ]:
df = pd.DataFrame(
    {
        "trade_date": pd.to_datetime(["2024-01-02", "2024-01-03"]),
        "settlement_date": pd.to_datetime(["2024-01-05", "2024-01-10"]),
    }
)
df["bdays"] = cal.bizdays(df["trade_date"], df["settlement_date"])
df

Results are positional NumPy arrays: the Series index is not kept. Assign a result straight to a column, as above. When you need a Series, pass `index=`.

In [ ]:
df2 = df.set_index(pd.Index(["a", "b"]))
pd.Series(cal.bizdays(df2["trade_date"], df2["settlement_date"]), index=df2.index)

Dates outside the calendar range raise `DateOutOfRange`. Check input dates first with `inrange`, and compute only the rows inside the range. `filled(False)` turns missing flags into `False`.

In [ ]:
loans = pd.DataFrame({"start": ["2024-01-02", "2024-01-03"], "maturity": ["2024-06-15", "2100-01-03"]})
ok = cal.inrange(loans.maturity).filled(False)
loans.loc[ok, "du"] = cal.bizdays(loans.start[ok], loans.maturity[ok])
loans

`getdate` also works with vectorized month references that come from pandas objects.

In [ ]:
months = pd.Index(["2024-01", "2024-02", "2024-03"])
cal.getdate("15th bizday", months)

Missing pandas inputs are preserved as `NaT` (dates), `NaN` (counts) or masked values (`isbizday`), depending on the method.

In [ ]:
cal.bizdays(pd.to_datetime(["2024-01-02", None]), pd.Timestamp("2024-01-05"))